The code below will work for the majority of tables. Upon running the code you will select whether you are taking the excel files from a website or your own computer. Once you input either a webpage URL or a file path you will be given the options of the excel files available. Once you select the files you wish to examine, you will then be given a list of the tabs available for processing and you can once again select only what you are interested in converting or you can also select "all". This will take you selection and convert the excel file(s) into a tidy data table format ready for further analysis. (Outputs will be found in a folder titled "tidy_outputs" found in the panel on the left.)


In [1]:
import os
import re
from io import BytesIO
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup


# --------------------------------
# SMALL UTILITIES
# --------------------------------
def format_filename(text):
    text = str(text).strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_")


def get_excel_basename(source):
    parsed = urlparse(str(source))
    if parsed.scheme in ("http", "https"):
        return os.path.basename(parsed.path)
    return os.path.basename(str(source))


def clean_path(path):
    # Handles Windows 'Copy as path' (usually includes quotes)
    return path.strip().strip('"').strip("'")


def is_blank(x):
    return pd.isna(x) or str(x).strip() == ""


def is_number_like(x):
    if pd.isna(x):
        return False
    if isinstance(x, (int, float)):
        return True

    s = str(x).strip()
    if s == "":
        return False

    s = s.replace(",", "").replace("%", "").replace("€", "")

    try:
        float(s)
        return True
    except:
        return False


def make_unique(columns):
    seen = {}
    new_cols = []
    for col in columns:
        col = str(col).strip()
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    return new_cols


def detect_header_rows(df, max_scan_rows=10, max_header_rows=4):
    """
    Detect likely header rows near the top of the dataframe.
    """
    header_rows = []

    for i in range(min(max_scan_rows, len(df))):
        row = df.iloc[i]
        values = [x for x in row if not is_blank(x)]
        if len(values) == 0:
            continue
        if len(values) == 1:  # likely title row
            continue

        text_cells = sum(not is_number_like(x) for x in values)
        text_ratio = text_cells / len(values)

        if text_ratio >= 0.6:
            header_rows.append(i)
            if len(header_rows) >= max_header_rows:
                break
        else:
            if header_rows:
                break

    return header_rows if header_rows else [0]


def clean_combined_headers(header_block):
    """
    Combine multiple header rows into one column name.
    """
    header_block = header_block.ffill(axis=1)

    new_columns = []
    for col in header_block.columns:
        parts = []
        for val in header_block[col]:
            if not is_blank(val):
                part = str(val).strip()
                if part not in parts:
                    parts.append(part)

        if parts:
            new_columns.append(" | ".join(parts))
        else:
            new_columns.append(f"column_{col}")

    return make_unique(new_columns)


def looks_like_footer(value):
    if pd.isna(value):
        return False

    s = str(value).strip().lower()
    footer_terms = [
        "coverage",
        "source",
        "note",
        "notes",
        "total coverage",
        "data source",
        "prepared by"
    ]
    return any(term in s for term in footer_terms)


def looks_like_date_or_period(value):
    if pd.isna(value):
        return False

    s = str(value).strip()

    if re.fullmatch(r"\d{4}", s):
        return True

    if re.fullmatch(r"Q[1-4]\s*\d{4}", s, flags=re.IGNORECASE):
        return True
    if re.fullmatch(r"\d{4}\s*Q[1-4]", s, flags=re.IGNORECASE):
        return True

    if re.fullmatch(r"[A-Za-z]{3,9}\s+\d{4}", s):
        return True

    try:
        pd.to_datetime(s, errors="raise")
        return True
    except:
        return False


def is_year_col(col):
    """
    Robust year detection for column headers:
    - matches 2010, "2010", 2010.0
    - matches "Income | 2010"
    """
    if pd.isna(col):
        return False

    if isinstance(col, (int, float)):
        try:
            y = int(col)
            return 1900 <= y <= 2100
        except:
            return False

    s = str(col).strip()
    if re.fullmatch(r"\d{4}", s):
        return True
    return bool(re.search(r"\b(19|20)\d{2}\b", s))


def extract_year_from_header(text):
    m = re.search(r"\b((?:19|20)\d{2})\b", str(text))
    return m.group(1) if m else None


def clean_numeric_series(series):
    s = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("€", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA})
    )
    return pd.to_numeric(s, errors="coerce")


def find_year_row_index(df, max_scan=25):
    """
    Find a row near the top containing many year-looking entries (e.g. 2010..2024).
    Returns (best_row_idx, best_year_count).
    """
    scan_rows = min(max_scan, len(df))
    best_row_idx = None
    best_year_count = 0

    for i in range(scan_rows):
        row_vals = [str(x).strip() for x in df.iloc[i].tolist()]
        year_count = sum(bool(re.fullmatch(r"(19|20)\d{2}", v)) for v in row_vals)
        if year_count > best_year_count:
            best_year_count = year_count
            best_row_idx = i

    return best_row_idx, best_year_count


# --------------------------------
# MAIN TIDY FUNCTION
# --------------------------------
def tidy_dataframe_to_long(df):
    """
    Handles three scenarios:
    1) Already-long tables with a Value column (keeps CoverType/Measure/etc.)   FIX
    2) Wide tables where years are columns (2010..2024 across top)
    3) Tables where dates/periods are in the first column (melt numeric columns only)
    """

    # Remove fully empty rows/cols
    df = df.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
    if df.empty:
        return df

    # Try to detect a "year row" early (helps stacked titles/merged headers)
    yr_idx, yr_count = find_year_row_index(df, max_scan=25)
    if yr_idx is not None and yr_count >= 3:
        promoted = df.iloc[yr_idx].tolist()
        new_cols = []
        for j, v in enumerate(promoted):
            v = str(v).strip()
            new_cols.append(v if v and v.lower() not in {"nan", "none"} else f"column_{j}")
        new_cols = make_unique(new_cols)

        df2 = df.iloc[yr_idx + 1:].reset_index(drop=True)
        if not df2.empty:
            df2.columns = [str(c).strip() for c in new_cols[: df2.shape[1]]]

            # If it now looks like "years across columns", handle that
            year_cols = [c for c in df2.columns if is_year_col(c)]
            if len(year_cols) >= 3:
                return wide_years_to_long(df2, year_cols)

    # Normal header detection & combine header rows
    header_rows = detect_header_rows(df)
    header_block = df.iloc[header_rows].copy()
    new_columns = clean_combined_headers(header_block)

    data_start = max(header_rows) + 1
    df = df.iloc[data_start:].reset_index(drop=True)
    if df.empty:
        return df

    df.columns = [str(c).strip() for c in new_columns]
    df = df.dropna(how="all").reset_index(drop=True)
    if df.empty:
        return df

    # -----------------------------
    # (1) ALREADY-LONG DETECTION 
    # If there is a "Value" column, keep all other columns as dimensions.
    # This prevents losing CoverType/Measure etc.
    # -----------------------------
    lower_to_orig = {c.lower(): c for c in df.columns}
    value_col = None
    for key in ["value", "values", "amount"]:
        if key in lower_to_orig:
            value_col = lower_to_orig[key]
            break

    if value_col is not None:
        # Clean numeric Value only
        df[value_col] = clean_numeric_series(df[value_col])
        df = df.dropna(subset=[value_col]).reset_index(drop=True)

        # Standardise name to 'value'
        if value_col != "value":
            df = df.rename(columns={value_col: "value"})

        return df

    # -----------------------------
    # (2) YEARS-AS-COLUMNS DETECTION
    # -----------------------------
    year_cols = [c for c in df.columns if is_year_col(c)]
    if len(year_cols) >= 3:
        return wide_years_to_long(df, year_cols)

    # -----------------------------
    # (3) DATE/PERIOD IN FIRST COLUMN (original idea)
    # Keep text columns as text; only melt numeric columns.
    # -----------------------------
    first_col = df.columns[0]

    # remove footer-like rows
    df = df[~df[first_col].apply(looks_like_footer)].reset_index(drop=True)
    if df.empty:
        return df

    df = df.rename(columns={first_col: "date"})

    # keep only plausible date/period rows
    df = df[df["date"].apply(looks_like_date_or_period)].reset_index(drop=True)
    if df.empty:
        return df

    other_cols = [c for c in df.columns if c != "date"]

    # Decide which columns are numeric enough to melt
    melt_cols = []
    for col in other_cols:
        sample = df[col].dropna().head(30)
        if len(sample) == 0:
            continue
        numeric_ratio = sum(is_number_like(x) for x in sample) / len(sample)
        if numeric_ratio >= 0.7:
            df[col] = clean_numeric_series(df[col])
            if df[col].notna().sum() > 0:
                melt_cols.append(col)
        else:
            # keep as text
            df[col] = df[col].astype(str).str.strip().replace({"": pd.NA})

    if not melt_cols:
        return pd.DataFrame()

    long_df = df.melt(id_vars=["date"], value_vars=melt_cols, var_name="variable", value_name="value")
    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)
    return long_df


def wide_years_to_long(df, year_cols):
    """
    Helper: convert 'years across columns' wide tables into long format.
    """
    non_year_cols = [c for c in df.columns if c not in year_cols]
    if not non_year_cols:
        return pd.DataFrame()

    # Rename first two non-year columns to stable names; preserve the rest
    rename_map = {non_year_cols[0]: "measure"}
    id_vars = ["measure"]

    if len(non_year_cols) >= 2:
        rename_map[non_year_cols[1]] = "measure_id"
        id_vars.append("measure_id")

    for k in range(2, len(non_year_cols)):
        dim_name = f"dimension_{k+1}"
        rename_map[non_year_cols[k]] = dim_name
        id_vars.append(dim_name)

    df = df.rename(columns=rename_map)

    # Remove footer-like rows
    df = df[~df["measure"].apply(looks_like_footer)].reset_index(drop=True)
    if df.empty:
        return pd.DataFrame()

    # Clean year columns
    for yc in year_cols:
        df[yc] = clean_numeric_series(df[yc])

    long_df = df.melt(id_vars=id_vars, value_vars=year_cols, var_name="year", value_name="value")
    long_df = long_df.dropna(subset=["value"]).reset_index(drop=True)

    # Extract year if embedded in header text
    long_df["year"] = long_df["year"].apply(extract_year_from_header)
    long_df = long_df.dropna(subset=["year"]).reset_index(drop=True)

    return long_df


# --------------------------------
# INPUT + SELECTION HELPERS
# --------------------------------
def choose_input_mode():
    print("Choose input method:")
    print("1. Scrape Excel files from a webpage")
    print("2. Use a single local Excel file")
    print("3. Use a folder of local Excel files")

    while True:
        choice = input("\nEnter 1, 2, or 3: ").strip()
        if choice in {"1", "2", "3"}:
            return choice
        print("Invalid choice.")


def choose_from_numbered_list(items, prompt):
    while True:
        try:
            choice = int(input(prompt).strip())
            if 1 <= choice <= len(items):
                return items[choice - 1]
            print("Please enter a valid number from the list.")
        except:
            print("Please enter a valid number.")


def choose_sheets(sheet_names):
    print("\nSheets available:")
    print("-" * 40)
    for i, name in enumerate(sheet_names, start=1):
        print(f"{i}. {name}")
    print("-" * 40)
    print(f"Total sheets: {len(sheet_names)}")

    # Save full list to file in case console truncates
    with open("sheet_list.txt", "w", encoding="utf-8") as f:
        for i, name in enumerate(sheet_names, start=1):
            f.write(f"{i}. {name}\n")
    print("Full sheet list saved to: sheet_list.txt")

    print("\nOptions:")
    print(" - Enter a single number (e.g. 2)")
    print(" - Enter multiple numbers separated by commas (e.g. 1,3,4)")
    print(" - Enter 'all' to process all sheets")

    while True:
        selection = input("\nSelect sheets: ").strip().lower()
        if selection == "all":
            return sheet_names

        try:
            indices = [int(x.strip()) for x in selection.split(",")]
            if all(1 <= i <= len(sheet_names) for i in indices):
                chosen = []
                for i in indices:
                    nm = sheet_names[i - 1]
                    if nm not in chosen:
                        chosen.append(nm)
                return chosen
            print("Invalid selection.")
        except:
            print("Please enter valid numbers or 'all'.")


# --------------------------------
# MAIN
# --------------------------------
def main():
    headers = {"User-Agent": "Mozilla/5.0"}
    mode = choose_input_mode()

    selected_source = None
    df_dict = None

    # MODE 1: Web scrape
    if mode == "1":
        page_url = input("\nInsert webpage link: ").strip()

        response = requests.get(page_url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        excel_links = []
        for link in soup.find_all("a"):
            href = link.get("href")
            if href and (".xlsx" in href.lower() or ".xlsm" in href.lower()):
                excel_links.append(urljoin(page_url, href))

        excel_links = list(dict.fromkeys(excel_links))
        if not excel_links:
            print("No Excel files found on the webpage.")
            return

        print("\nExcel files found:\n")
        for i, link in enumerate(excel_links, start=1):
            print(f"{i}. {get_excel_basename(link)}")

        selected_source = choose_from_numbered_list(excel_links, "\nSelect file number: ")

        file_data = requests.get(selected_source, headers=headers)
        file_data.raise_for_status()

        df_dict = pd.read_excel(BytesIO(file_data.content), sheet_name=None, engine="openpyxl")

    # MODE 2: Local file
    elif mode == "2":
        selected_source = clean_path(input("\nEnter full file path: "))
        if not os.path.exists(selected_source):
            print("File not found. Check the path you pasted (quotes are OK).")
            return

        df_dict = pd.read_excel(selected_source, sheet_name=None, engine="openpyxl")

    # MODE 3: Folder
    elif mode == "3":
        folder_path = clean_path(input("\nEnter folder path: "))
        if not os.path.isdir(folder_path):
            print("That folder path does not exist.")
            return

        files = [f for f in os.listdir(folder_path) if f.lower().endswith((".xlsx", ".xlsm"))]
        files.sort()
        if not files:
            print("No Excel files (.xlsx/.xlsm) found in that folder.")
            return

        print("\nExcel files available:\n")
        for i, f in enumerate(files, start=1):
            print(f"{i}. {f}")

        chosen_file = choose_from_numbered_list(files, "\nSelect file number: ")
        selected_source = os.path.join(folder_path, chosen_file)

        df_dict = pd.read_excel(selected_source, sheet_name=None, engine="openpyxl")

    # Sheet selection
    sheet_names = list(df_dict.keys())
    selected_sheets = choose_sheets(sheet_names)
    print("\nSelected sheets:", selected_sheets)

    # Processing + output
    os.makedirs("tidy_outputs", exist_ok=True)

    base_name = os.path.splitext(get_excel_basename(selected_source))[0]
    file_part = format_filename(base_name)

    for sheet_name in selected_sheets:
        print("\nTidying sheet:", sheet_name)
        df = df_dict[sheet_name]

        try:
            tidy_df = tidy_dataframe_to_long(df)

            if tidy_df is None or tidy_df.empty:
                print("  Skipped (empty after tidying)")
                continue

            sheet_part = format_filename(sheet_name)
            output_name = f"{file_part}__{sheet_part}.csv"

            tidy_df.to_csv(os.path.join("tidy_outputs", output_name), index=False)

            print("  Saved:", output_name)
            print("  Output columns:", list(tidy_df.columns))

        except Exception as e:
            print("  Error tidying sheet:", sheet_name, "-", e)

    print("\nFinished processing.")


if __name__ == "__main__":
    main()


Choose input method:
1. Scrape Excel files from a webpage
2. Use a single local Excel file
3. Use a folder of local Excel files

Excel files found:

1. data-annex-ncid-private-motor-insurance-mid-year-2025-settled.xlsx
2. data-annex-ncid-private-motor-insurance-mid-year-2025.xlsx
3. annex-private-motor-insurance-report-7.xlsx
4. data-annex-ncid-private-motor-insurance-mid-year-2024-settled-claims.xlsx
5. data-annex-ncid-private-motor-insurance-mid-year-2024.xlsx
6. annex-ncid-private-motor-insurance-report-6.xlsx
7. data-annex-ncid-private-motor-insurance-mid-year-report-2.xlsx
8. annex-ncid-private-motor-insurance-report-5.xlsx
9. data-annex-ncid-private-motor-insurance-mid-year-report-1.xlsx
10. annex-private-motor-insurance-report-4.xlsx
11. annex-private-motor-insurance-report-3.xlsx
12. annex-private-motor-insurance-report-2-data.xlsx
13. annex-2-private-motor-insurance-report-2.xlsx
14. annex-1---settlement-channels-2015-to-2018.xlsx
15. annex-2---private-motor-insurance-report-1